# ML-08 — Logistic Regression vs. Refresh Baseline

This notebook tests whether a Logistic Regression ranking improves on the Week-4 transparent refresh rule. It uses the starter snapshot and its current-state decline proxy only for practice. It does not claim to predict a future outcome or prove that a refresh causes recovery.

## 1. Method choice and why

I use **Logistic Regression** because the temporary outcome has two classes: `down` or not `down`. The model returns a probability of the `down` class, which can be sorted into a review queue. It is a suitable first model because its coefficients are inspectable and it is less complex than a tree ensemble. The baseline remains the simple rule: pages stale for 180+ days and with 500+ impressions receive a refresh score.

The outcome is `decline_proxy_rule = (trend_direction == 'down')`. It is a current-state proxy, not an observed future label. `trend_direction`, `trend_pct`, and the recent-window fields that construct the trend are all excluded from model features.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists())
output_dir = repo_root / 'work/outputs'
output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(repo_root / 'data/raw/content_refresh_anonymized.csv')
df['decline_proxy_rule'] = df['trend_direction'].eq('down').astype(int)
print(f'Rows: {len(df):,}; clients: {df.client_id.nunique()}; proxy decline rate: {df.decline_proxy_rule.mean():.1%}')
print(f'pandas {pd.__version__}; scikit-learn {sklearn.__version__}; seed {RANDOM_SEED}')


Rows: 30,000; clients: 32; proxy decline rate: 54.2%
pandas 2.0.3; scikit-learn 1.3.2; seed 42


## 2. Split design

I use a **grouped client split**: 75% of pseudonymized clients for training and 25% different clients for testing. `client_id` is used only to make the split; it is not a model feature. This is more honest than randomly mixing pages from the same client into both sets, because client-level patterns cannot simply be memorized. The baseline and model are both evaluated on exactly this test set.

In [2]:
feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'impressions_90d',
    'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
# Excluded: identifiers, trend_direction, trend_pct, and all last/previous-30-day trend inputs.
target_col = 'decline_proxy_rule'
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(df, df[target_col], groups=df['client_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

split_summary = pd.DataFrame([
    ['train', len(train_df), train_df.client_id.nunique(), train_df[target_col].mean()],
    ['test', len(test_df), test_df.client_id.nunique(), test_df[target_col].mean()],
] , columns=['split', 'rows', 'clients', 'proxy_decline_rate'])
split_summary['proxy_decline_rate'] = split_summary['proxy_decline_rate'].map(lambda x: f'{x:.1%}')
split_summary

,split,rows,clients,proxy_decline_rate
0,train,22885,24,55.0%
1,test,7115,8,51.7%


## 3. Train + compare versus my baseline

Both methods rank the **same held-out test pages**, and both are scored with Precision@10 and Precision@20 against the same audit proxy. The base rate is shown so the top-K results have context. The model is trained only on training clients.

The baseline uses only the Week-4 rule: stale (180+ days) and visible (500+ impressions), with a score that increases with impressions and staleness.

In [3]:
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ]), feature_cols),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('logistic', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)),
])
model.fit(X_train, y_train)
test_df['model_probability'] = model.predict_proba(X_test)[:, 1]

eligible = (test_df['days_since_last_update'] >= 180) & (test_df['impressions_90d'] >= 500)
test_df['baseline_score'] = np.where(
    eligible,
    np.log1p(test_df['impressions_90d']) * (1 + test_df['days_since_last_update'] / 365),
    0.0,
)

def precision_at_k(frame, score_col, k):
    return frame.sort_values(score_col, ascending=False).head(k)[target_col].mean()

comparison = pd.DataFrame([
    ['Week-4 rule baseline', precision_at_k(test_df, 'baseline_score', 10), precision_at_k(test_df, 'baseline_score', 20), np.nan, np.nan],
    ['Logistic Regression', precision_at_k(test_df, 'model_probability', 10), precision_at_k(test_df, 'model_probability', 20),
     average_precision_score(y_test, test_df['model_probability']), roc_auc_score(y_test, test_df['model_probability'])],
] , columns=['method', 'precision_at_10', 'precision_at_20', 'average_precision', 'roc_auc'])
comparison['test_base_rate'] = y_test.mean()
for column in ['precision_at_10', 'precision_at_20', 'average_precision', 'roc_auc', 'test_base_rate']:
    comparison[column] = comparison[column].map(lambda value: '' if pd.isna(value) else f'{value:.3f}')
comparison

with open(output_dir / 'w05_model_metrics.json', 'w', encoding='utf-8') as handle:
    json.dump({
        'seed': RANDOM_SEED, 'train_rows': int(len(train_df)), 'test_rows': int(len(test_df)),
        'train_clients': int(train_df.client_id.nunique()), 'test_clients': int(test_df.client_id.nunique()),
        'features': feature_cols, 'excluded_leakage_fields': ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d'],
        'comparison': comparison.to_dict(orient='records')
    }, handle, indent=2)

print('Model-versus-baseline comparison on the held-out test clients:')
display(comparison)


Model-versus-baseline comparison on the held-out test clients:


,method,precision_at_10,precision_at_20,average_precision,roc_auc,test_base_rate
0,Week-4 rule baseline,0.800,0.700,,,0.517
1,Logistic Regression,0.900,0.850,0.600,0.589,0.517


## 4. Errors and interpretation

The coefficients below are standardized logistic-regression coefficients. A positive coefficient means that, holding the other included signals constant, a higher feature value pushes the proxy-down probability upward. This is association within the starter snapshot, not a causal explanation.

In [4]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['logistic'].coef_[0]
importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
importance['abs_coefficient'] = importance['coefficient'].abs()
importance = importance[~importance['feature'].str.contains('missingindicator')].sort_values('abs_coefficient', ascending=False)
print('Top three signals by absolute standardized coefficient:')
display(importance.head(3)[['feature', 'coefficient']])

test_df['model_prediction'] = (test_df['model_probability'] >= 0.5).astype(int)
wrong = test_df.loc[test_df['model_prediction'] != test_df[target_col], [
    target_col, 'model_probability', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'days_since_last_update', 'engagement_rate', 'content_type'
]].copy()
wrong['error_type'] = np.where(wrong[target_col].eq(1), 'false negative', 'false positive')
wrong['confidence_distance'] = (wrong['model_probability'] - 0.5).abs()
wrong = wrong.sort_values('confidence_distance', ascending=False)
print(f'Classification errors at probability threshold 0.50: {len(wrong):,} of {len(test_df):,} test pages.')
display(wrong.head(3))

error_summary = wrong.groupby('error_type').agg(
    n=(target_col, 'size'),
    median_probability=('model_probability', 'median'),
    median_impressions=('impressions_90d', 'median'),
    median_position=('avg_position', 'median')
).reset_index()
display(error_summary)

print('Interpretation: the top coefficients show which observed snapshot signals the model leans on. The three displayed mistakes are difficult because a current snapshot can look strong while still being labelled down, or look weak while not being labelled down. The excluded trend fields would make the result suspiciously easy, so they remain excluded.')

Top three signals by absolute standardized coefficient:


,feature,coefficient
7,numeric__days_with_impressions,0.670979
8,numeric__days_with_sessions,-0.492503
9,numeric__content_age_days,-0.218729


Classification errors at probability threshold 0.50: 3,103 of 7,115 test pages.


,decline_proxy_rule,model_probability,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,engagement_rate,content_type,error_type,confidence_distance
20604,1,0.065397,6,0,0.0,12.2,20,0.0,keyword article,false negative,0.434603
13922,1,0.066588,13,0,0.0,11.5,20,0.0,keyword article,false negative,0.433412
4147,1,0.071110,13,0,0.0,33.2,20,0.0,keyword article,false negative,0.428890


,error_type,n,median_probability,median_impressions,median_position
0,false negative,1181,0.377832,37.0,10.4
1,false positive,1922,0.626943,773.0,10.1


Interpretation: the top coefficients show which observed snapshot signals the model leans on. The three displayed mistakes are difficult because a current snapshot can look strong while still being labelled down, or look weak while not being labelled down. The excluded trend fields would make the result suspiciously easy, so they remain excluded.


## 5. Self-check

- [x] I chose Logistic Regression because the practice proxy is binary and the output is a ranked probability.
- [x] I used a reproducible grouped-client split; client ID is never a feature.
- [x] Baseline and model are evaluated on the same test clients with the same Precision@10 and Precision@20 metrics.
- [x] I reported the test base rate and model ranking metrics.
- [x] I inspected coefficients and concrete model errors.
- [x] I excluded `trend_direction`, `trend_pct`, and the recent-window trend inputs that would leak the proxy.
- [x] The notebook was executed top to bottom; results are decision-support practice only.